# 🏅 第 3 阶段：结构化输出（Tool Calling 前置）

## 🎯 阶段目标

让模型输出"可解析的结构化数据"，而不仅仅是自然语言。

> **结构化输出 = JSON 格式 + 严格约束**

这是实现自动工具调用的关键前置技能。

---

## 🧠 核心概念

在这个阶段，我们需要让模型输出：

```json
{
  "thought": "用户问北京天气，我需要调用天气查询工具",
  "action": "get_weather",
  "params": {"city": "北京"}
}
```

---

## 📦 1. 加载配置

In [1]:
import json
import requests
import re
from pathlib import Path

# 加载配置
config_path = Path.cwd() / ".." / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_CONFIG = config["model"]

print("✅ 配置加载成功")

✅ 配置加载成功


---

## 🚀 2. 定义 LLM 调用函数

In [2]:
def call_llm(messages: list) -> str:
    """调用 LLM 模型"""
    url = f"{MODEL_CONFIG['base_url']}/v1/chat/completions"
    
    payload = {
        "model": MODEL_CONFIG['name'],
        "messages": messages,
        "temperature": 0.0,  # 低温度保证输出稳定
        "max_tokens": MODEL_CONFIG.get('max_tokens', 200)
    }
    
    try:
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]
    except Exception as e:
        return f"❌ 错误：{str(e)}"

print("✅ LLM 调用函数定义完成")

✅ LLM 调用函数定义完成


---

## 🔧 3. 定义 JSON 解析函数

In [3]:
def extract_json(response: str) -> dict:
    """
    从模型响应中提取 JSON
    """
    try:
        # 方法1：直接解析
        return json.loads(response)
    except json.JSONDecodeError:
        # 方法2：正则提取 JSON 块
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                return None
        return None

def validate_action(structured_data: dict) -> tuple:
    """
    验证结构化输出是否符合要求
    """
    if structured_data is None:
        return False, "输出不是有效的 JSON"
    
    required_fields = ["thought", "action", "params"]
    if not all(field in structured_data for field in required_fields):
        return False, f"缺少必要字段，当前字段: {list(structured_data.keys())}"
    
    valid_actions = ["get_weather", "calculate", "get_time", "finish"]
    if structured_data["action"] not in valid_actions:
        return False, f"无效的 action: {structured_data['action']}"
    
    return True, "验证通过"

print("✅ JSON 解析和验证函数定义完成")

✅ JSON 解析和验证函数定义完成


---

## 🎯 4. 定义结构化输出函数

In [4]:
def get_structured_output(user_input: str) -> dict:
    """
    获取模型的结构化输出
    """
    system_prompt = """
你是一个工具调用专家。请根据用户的问题，决定是否需要调用工具，并输出结构化的 JSON 响应。

可用工具：
1. get_weather(city): 获取指定城市的天气
2. calculate(expression): 数学计算
3. get_time(): 获取当前时间

输出格式（必须是有效的 JSON，只能包含以下字段）：
{
  "thought": "你的思考过程",
  "action": "工具名称或 'finish'",
  "params": {"参数名": "参数值"}
}

注意：
- action 只能是 get_weather、calculate、get_time 或 finish
- params 必须是对象类型
- 如果不需要调用工具，action 填 "finish"
"""

    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_input}
    ]
    
    response = call_llm(messages)
    print(f"📝 模型原始输出:\n{response}\n")
    
    # 解析 JSON
    structured_data = extract_json(response)
    
    if structured_data:
        # 验证
        valid, message = validate_action(structured_data)
        if valid:
            print("✅ 结构化输出验证通过:")
            print(json.dumps(structured_data, indent=2, ensure_ascii=False))
            return structured_data
        else:
            print(f"❌ 验证失败: {message}")
            return None
    else:
        print("❌ 无法解析结构化输出")
        return None

print("✅ 结构化输出函数定义完成")

✅ 结构化输出函数定义完成


---

## 🎮 5. 测试结构化输出

In [5]:
# 测试 1：天气查询
print("\n=== 测试 1：天气查询 ===")
result = get_structured_output("北京今天天气怎么样？")


=== 测试 1：天气查询 ===
📝 模型原始输出:
用户询问北京今天的天气情况，这是一个需要调用天气查询工具的问题。

根据可用工具列表，我应该使用 get_weather(city) 工具来获取北京的天气信息。

让我构建正确的 JSON 响应：
- thought: 说明需要查询北京天气
- action: get_weather
- params: {"city": "北京"}
</think>

```json
{
  "thought": "用户询问北京今天的天气情况，需要调用天气查询工具获取信息",
  "action": "get_weather",
  "params": {
    "city": "北京"
  }
}
```

❌ 无法解析结构化输出


In [6]:
# 测试 2：数学计算
print("\n=== 测试 2：数学计算 ===")
result = get_structured_output("1 + 2 等于多少？")


=== 测试 2：数学计算 ===
📝 模型原始输出:
用户问了一个简单的数学计算问题"1 + 2 等于多少？"。我需要使用 calculate 工具来计算这个表达式。

让我按照要求的格式输出 JSON 响应。
</think>

{
  "thought": "用户询问1 + 2的计算结果，这是一个数学计算问题，需要使用calculate工具来完成。",
  "action": "calculate",
  "params": {
    "expression": "1 + 2"
  }
}

✅ 结构化输出验证通过:
{
  "thought": "用户询问1 + 2的计算结果，这是一个数学计算问题，需要使用calculate工具来完成。",
  "action": "calculate",
  "params": {
    "expression": "1 + 2"
  }
}


In [7]:
# 测试 3：不需要工具
print("\n=== 测试 3：不需要工具 ===")
result = get_structured_output("你好，我是张三")


=== 测试 3：不需要工具 ===
📝 模型原始输出:
用户只是做了自我介绍，没有提出任何需要工具帮助的问题。这是一个简单的问候场景，不需要调用任何工具。

我应该输出一个JSON响应，action设为"finish"，params设为空对象，thought说明不需要调用工具的原因。
</think>

{
  "thought": "用户只是做了自我介绍，没有提出任何需要工具帮助的问题或请求。这是一个简单的问候场景，不需要调用任何工具。",
  "action": "finish",
  "params": {}
}

✅ 结构化输出验证通过:
{
  "thought": "用户只是做了自我介绍，没有提出任何需要工具帮助的问题或请求。这是一个简单的问候场景，不需要调用任何工具。",
  "action": "finish",
  "params": {}
}


---

## ✅ 本阶段总结

你已经完成了：

1. ✅ 学会了如何让模型输出结构化 JSON
2. ✅ 实现了 JSON 解析和验证
3. ✅ 掌握了 Prompt 工程技巧

---

## 🔗 下一步

进入 **[第 4 阶段：单步 Agent](../step4/step4.ipynb)**，实现模型自动决定是否调用工具。